<a href="https://colab.research.google.com/github/chijindumu/hash_pareto/blob/isscc27-submission/submission/ISSCC27/submitted_notebooks/hashpareto/hashpareto.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**IEEE SSCS Code-a-Chip ISSCC 2027 · Submission notebook** ·
Apache-2.0 · Runs top-to-bottom in Colab (Tier 0) or locally (Tiers 1–2).

In [ ]:
# @title Install core analysis deps {display-mode: "form"}
import subprocess, sys, os

def pip_install(pkgs):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + pkgs)

# pandas, matplotlib, ipywidgets are preinstalled on Colab; ensure here anyway.
pip_install(['pandas>=2.0', 'matplotlib>=3.7', 'ipywidgets>=8'])

# RTL verification needs Icarus or Verilator (apt on Debian/Colab, brew on macOS).
# The notebook's Tier 0 needs only pandas + matplotlib.
print('install ok')

In [ ]:
# Print tool versions (recommended by Code-a-Chip howtoapply.md).
import platform, shutil, subprocess, sys

def v(cmd):
    try:
        return subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout.strip().splitlines()[0]
    except Exception:
        return 'n/a'

print('Python   :', sys.version.split()[0], '|', platform.platform())
for name, cmd in [
    ('pandas',     "python3 -c 'import pandas; print(pandas.__version__)'"),
    ('matplotlib', "python3 -c 'import matplotlib; print(matplotlib.__version__)'"),
    ('iverilog',   'iverilog -V 2>&1 | head -1'),
    ('verilator',  'verilator --version'),
    ('yosys',      'yosys -V 2>&1 | head -1'),
    ('openroad',   'openroad -version 2>&1 | head -1'),
    ('librelane',  "python3 -c 'import librelane; print(librelane.__version__)'"),
]:
    print(f'{name:12s}:', v(cmd))

# HashPareto — a notebook-driven PPA explorer for an open SHA-256 core

```text
Copyright 2026 Benjamin Joshua Chimbuchi
SPDX-License-Identifier: Apache-2.0
```

| Name | Affiliation | IEEE Member | SSCS Member | Email |
|------|-------------|------------|-------------|-------|
| Benjamin Joshua Chimbuchi | University of Ibadan | No | No | joshuabenjamin852@gmail.com |

## Abstract
>**Unrolling the SHA-256 round loop is the textbook area↔throughput knob.** We test this on an open RTL-to-GDS flow (LibreLane, SKY130) with a parameterized core (UNROLL ∈ {1,2,4,8,16}). We replace the usual closed-form model (`64/UNROLL` cycles/block) with **cycle-accurate measurements** of the synthesized RTL, which show the core becomes **input-bound at ≈18 cycles/block** (32-bit stream interface) for UNROLL ≥ 4. Hardening each variant at its own **closed-timing clock** reveals that unrolling beyond 4× incurs area overhead without throughput benefit — the **knee of the Pareto curve is U4**. All results (verification, cycle measurement, synthesis, Pareto analysis) are reproducible from committed data; the full flow runs on WSL2+LibreLane+SKY130. This notebook provides a **three-tier framework** (instant verification, measured performance, opt-in full flow) so judges can engage at their preferred depth.


## 1. Environment

The whole flow runs under **WSL2 (Ubuntu 26.04)** on Windows 11. LibreLane
is distributed as a Nix-based **devshell AppImage**, so there is no Docker
image to pull and no root install. The SKY130 PDK is fetched once with CIEL
into `~/.ciel/`. Exact versions and SHA-256 hashes of every tool are pinned
in [`docs/phase0_environment.md`](docs/phase0_environment.md).

A Python virtualenv (`~/venv`) carries cocotb, pandas, matplotlib, and
Jupyter. All commands are plain shell; from Windows prefix with
`wsl -d Ubuntu-26.04 -- bash -lc "..."`.

**Recommended tool versions.** These are the exact versions this notebook
was produced and last verified with; anything close should work, but pin
these for bit-identical knockout logs:

| Tool | Version | Role | License |
|------|---------|------|---------|
| Ubuntu (WSL2) | 26.04.1 LTS | host OS | — |
| Python (venv) | 3.14.4 | notebook kernel | PSF |
| JupyterLab | 4.6.4 | notebook runtime | BSD-3 |
| LibreLane | 3.0.16 (devshell AppImage) | RTL→GDS flow | Apache-2.0 |
| Yosys | 0.62 | synthesis | ISC |
| OpenROAD | 1.0 (LibreLane-pinned) | PnR + STA | Apache-2.0 |
| Magic | 8.3.623 | DRC / ext | MIT-style |
| Netgen | 1.5.316 | LVS | MIT-style |
| KLayout | 0.30.7 | GDS view / PNG render | GPL-3.0 |
| Verilator | 5.044 | RTL simulation | LGPL-3 / Artistic-2 |
| cocotb | 2.1.0 | testbench framework | BSD-3 |
| pandas / matplotlib | 3.0.6 / 3.11.2 | analysis + plots | BSD-3 / PSF |
| SKY130 PDK | sky130A (CIEL, 2024-ish) | process | Apache-2.0 |

## 2. Related work

**SHA-256 hardware.** The compression core follows FIPS 180-4 [1]; its
round recurrence is the classic Maj/Ch/Σ construction. Unrolling is the
standard area↔throughput knob and appears in most published SHA-256 ASIC
designs, so this notebook deliberately varies *only* that one parameter
and hardens every point in the same PDK with the same flow — a comparison
the literature rarely makes because most papers report a single design
under one vendor toolchain. The upstream `nano_hash` core [2] supplies the
iterative baseline and the AXI-Stream shell.

**Open-source RTL-to-GDS flows.** OpenLane 2 / LibreLane [3] wrap Yosys,
OpenROAD, Magic and Netgen into a reproducible 77-step flow that runs on a
laptop, with SKY130 [4] as the open PDK. That combination is what makes a
notebook like this possible at all: no commercial license, no NDA, and a
GDS at the end.

**Notebook-driven chip design.** Prior Code-a-Chip winners [5] and the
reference inverter / temperature-sensor notebooks [6][7] established the
format: a single Jupyter file that explains the idea, runs the flow, and
shows the results. This project follows that format for a *design-space
exploration* rather than a single circuit or a generator: the contribution
is the measured Pareto curve, plus an honest negative result (U8).

> The numbered links resolve in §11.

### Prior work vs new in this submission

| | Pre-existing (`nano_hash`, Apache-2.0, vendored under `rtl/original/`) | New in HashPareto |
|:--|:--|:--|
| SHA-256 compression core | Iterative 64-round core (`hashing.sv`, `sha_core.sv`) | **Parameterized rewrite** `sha256_core.sv` with `UNROLL` knob |
| Message padding / digest | Vendored `message_padder.sv`, `digest_buffer.sv` — reused unchanged | Reused unchanged (shared shell) |
| Top / handshake | Vendored `sha_256_top.sv` | Rewritten `sha_top.sv` with same AXI-Stream port list |
| Verification | Nano_hash testbenches | **cocotb + Verilator** matrix: 5 suites × 5 variants = 25/25 PASS vs `hashlib` |
| Physical design | None | **LibreLane hardening** of all variants on SKY130, GDS layouts, Pareto analysis |
| Performance model | Closed-form `64/U` | **Cycle-accurate measurement** (67/35/19/18/18 cycles/block) |
| Notebook | None | This notebook: Colab-ready, three tiers, live data, layout renders |

## 3. The design: one core, five variants

The starting point is `nano_hash` (vendored under `rtl/original/`,
Apache-2.0), a clean 64-round iterative SHA-256 compression core.
HashPareto's contribution is the **parameterized rewrite**:

- `rtl/generated/sha256_core.sv` — one module with `parameter int UNROLL`.
  Each cycle it evaluates `UNROLL` SHA-256 rounds, so a 512-bit block takes
  `NCHUNK = 64 / UNROLL` compression cycles (64, 32, 16, 8, or 4). The message schedule
  `W[t]` is extended *in-cycle* through an `ext[]` chain rather than a
  16-word rotating register — this keeps the schedule combinational and
  correct at every unroll factor without duplicating the RAM-style storage.
- `rtl/generated/sha_top.sv` — same AXI-Stream port list as the original top;
  reuses the **untouched** vendored `message_padder` and `digest_buffer`, so
  the only new logic under test is the compression core.
- A `kconst()` function returns the 64 round constants (a `case` statement —
  this sidesteps an Icarus limitation with unpacked `localparam` arrays and
  is trivially constant-folded by Yosys).

The unroll factor is the *only* knob. Everything else — padding, digest
buffering, handshake — is shared, so differences in the hardened results
are cleanly attributable to the unrolling decision.

```systemverilog
module sha256_core #(parameter int UNROLL = 1) ( ... );
    localparam int NCHUNK = 64 / UNROLL;   // compression cycles per 512-bit block (model)
    // ... UNROLL rounds evaluated per clock, in-cycle W[] extension ...
endmodule
```

## 4. Functional verification: 25/25 bit-exact

Before any silicon work, every variant is proven bit-exact against Python's
`hashlib.sha256` (the gold standard) with a **cocotb + Verilator** harness
(`verif/cocotb/`). Five suites run on all five variants:

| # | Suite | What it covers |
|--:|-------|----------------|
| 1 | Known vectors | empty, `"abc"`, NIST 56- and 112-byte vectors, `bytes(range(256))`, **1 000 000 × `'a'`** (≈15 625 blocks, stresses multi-block chaining) |
| 2 | Boundary lengths | 28 lengths straddling the 55/56 and 63/64 packing boundaries |
| 3 | Random | 40 randomized messages, 0–300 bytes |
| 4 | Ingress bubbles | randomized `s_axis_tvalid` deassertions |
| 5 | Egress backpressure | randomized `m_axis_tready` stalls mid-digest |

**Result: 5 variants × 5 suites = 25/25 PASS (0 failures).**
Full method: [`docs/phase1_verification.md`](docs/phase1_verification.md).

Re-run it yourself (~2 min, needs the toolchain from §1):

```bash
for u in 1 2 4 8 16; do ~/venv/bin/python verif/cocotb/run.py --sim verilator --unroll $u; done
```

## 5. Physical implementation: LibreLane on SKY130

Each variant gets a flow directory (`flow/variants/u<N>/`) with a Yosys-ready
`src/` and a four-line `config.yaml`:

```yaml
DESIGN_NAME: sha_top
VERILOG_FILES: dir::src/*.sv
CLOCK_PORT: aclk
CLOCK_PERIOD: 100.0        # 300.0 for U16
```

One command takes it to GDS (a 77-step flow: synthesis → floorplan →
placement → CTS → global + detailed routing → parasitic extraction → STA →
DRC/LVS/antenna sign-off → GDS):

```bash
tools/run_variant.sh <UNROLL> <run-tag>      # e.g. tools/run_variant.sh 4 grid100
```

### Clocking strategy (hybrid)

U1, U2, U4 all close timing at a **100 ns** period (10 MHz) with zero slack
violations. U16 — 16 rounds in one cycle — has a *combinational depth* of ~16
chained majority/XOR/adder stages; its pre-CTS critical path alone was
**132 ns**, and OpenROAD's repair loop could not converge on 100 ns in a
reasonable runtime. Rather than report a broken point, we run U16 at **300 ns**
and say so explicitly. This is why the throughput curve flattens at high unroll
(§7).

### Sign-off status

All four hardened variants pass: **DRC = 0, LVS clean, antenna = 0,
setup WNS = 0 ns** at the chosen clock (worst-negative slack metric). Per-corner **worst setup slack** and estimated fmax are reported below; slew/cap/fanout violations remain in some non-nominal corners (documented). An earlier deliberately tight run (U1 @ 10 ns) failed timing — evidence that 100 ns is conservative for U1/U2/U4.
over-aggressive run (U1 @ 10 ns) failed setup (WNS = −9.48 ns) and is kept in
`data/metrics.csv` as a counter-example of what happens when you ignore STA.

U8 (UNROLL = 8 @ 100 ns) hit **routing congestion** (`GRT-0116`) at global
routing; it is reported as a limitation in §9 rather than silently dropped.

### Loading the measured metrics

This cell locates the committed metrics directory — `data/` in the submitted
package, `results/` in the development repository — by walking up from the
current directory, then loads `metrics.csv` (one row per finished LibreLane
run). It prints the directory it chose, and you should see a raw table with
area, power, timing slack, and DRC/antenna counts for each run. The
deliberately-broken 10 ns counter-example is kept in the raw table but
dropped from the Pareto math below.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

def find_data():
    """Locate the small metrics dir: data/ (submission) or results/ (dev repo).
    Searches the CWD and all its parents so the notebook works whether it is
    opened at the package root, in notebook/, or from any subdirectory."""
    for base in [Path.cwd(), *Path.cwd().parents]:
        for name in ('data', 'results'):
            d = base / name
            if (d / 'metrics.csv').exists():
                return d
    raise FileNotFoundError('metrics.csv not found in any data/ or results/ dir')

DATA = find_data()
print('data dir :', DATA.resolve())

# Raw per-run metrics (one row per finished LibreLane run), incl. the 10 ns counter-example.
dfm = pd.read_csv(DATA / 'metrics.csv')
# Drop the deliberately-violated 10 ns run from the Pareto math; keep it in the table.
dfm_ok = dfm[dfm['run'] != 'run_u1'].copy()
cols = ['unroll', 'run', 'clock_period_ns', 'area_stdcell_um2', 'cells_stdcell',
        'power_W', 'setup_wns_ns', 'drc_errors', 'antenna_violations']
dfm_ok[cols]

## 6. Throughput and energy model

With the raw metrics in hand, the derived quantities are simple:

- **Cycles per 512-bit block** (model) = `64 / UNROLL` (SHA-256 has 64 rounds). In steady state with the 32-bit ingest port, measured cycles/block are **67, 35, 19, 18, 18** for U1–U16 (RTL simulation).
- **Time per block** = `cycles_per_block × clock_period`
- **Throughput** = `1 / time_per_block`  (blocks/s = hashes/s)
- **Energy per hash** = `power_total × time_per_block`

`power_total` is the vectorless STA power reported by OpenROAD at the operating
corner — it includes internal, switching, and leakage power of the stdcell
network. This is a standard first-order estimate; it does not model I/O pad
power or clock-tree activity beyond what the STA assumes.

Now the arithmetic: for every row we compute cycles per block, time per
block, throughput, and energy, building the derived Pareto table from first
principles. The printed cross-check must read **0.000 %** — that is this
cell independently reproducing the committed `pareto.csv` rather than
trusting it. The last line renders the table you should see below.

In [ ]:
# Recompute the Pareto table from first principles using MEASURED cycles.
# Step 1: load measured cycles/block (RTL simulation, not the closed-form 64/U).
try:
    cdf = pd.read_csv(DATA / 'cycles.csv')
except Exception:
    cdf = pd.DataFrame(columns=['unroll', 'cycles_per_block'])

# Step 2: pick ONE representative run per variant. Prefer timing-closed
# (setup_ws_worst_ns >= 0); otherwise pick the tightest clock we have,
# and flag it so the reader knows it's an open-timing point.
cand = []
for u, g in dfm_ok.groupby('unroll'):
    closed = g[g['setup_ws_worst_ns'].astype(float) >= 0]
    pick = closed.iloc[[closed['clock_period_ns'].astype(float).idxmin()]] if len(closed) else g.iloc[[g['clock_period_ns'].astype(float).idxmin()]]
    pick = pick.copy()
    pick['timing_closed'] = len(closed) > 0
    cand.append(pick)
dfm_best = pd.concat(cand).sort_values('unroll').reset_index(drop=True) if cand else pd.DataFrame()

derived = []
for _, r in dfm_best.iterrows():
    u = int(r['unroll'])
    mrow = cdf[cdf['unroll'] == u]
    cycles = float(mrow['cycles_per_block'].iloc[0]) if len(mrow) else 64.0 / u
    t_blk = cycles * float(r['clock_period_ns']) * 1e-9          # s per 512-bit block
    derived.append({
        'unroll': u,
        'run': r['run'],
        'clock_ns': r['clock_period_ns'],
        'timing_closed': bool(r['timing_closed']),
        'area_um2': r['area_stdcell_um2'],
        'cells': int(r['cells_stdcell']),
        'power_mW': r['power_W'] * 1e3,
        'cycles_per_block': cycles,
        'throughput_Mhash_s': (1 / t_blk) / 1e6,
        'energy_nJ': (r['power_W'] * t_blk) * 1e9,
    })
dfp = pd.DataFrame(derived).sort_values('unroll').reset_index(drop=True)

# Cross-check against the committed pareto.csv (should match closely).
try:
    ref = pd.read_csv(DATA / 'pareto.csv')
    chk = dfp.merge(ref, on='unroll', suffixes=('_live', '_ref'))
    dE = (100 * (chk['energy_nJ'] - chk['energy_per_hash_nJ']).abs() / chk['energy_per_hash_nJ']).max()
    print('max energy deviation vs committed pareto.csv: %.3f %%' % dE)
except Exception as e:
    print('cross-check skipped:', e)

dfp.round(3)

The next two cells draw the Pareto curves. Read them as a pair — the first
prices the area you pay, the second prices the energy you get back. Each
point is labelled `U<unroll>`, and the dashed line between points is only a
visual guide (nothing was measured between them).

In [ ]:
# Area vs throughput.
fig, ax = plt.subplots(figsize=(6.4, 4.4))
ax.plot(dfp.throughput_Mhash_s, dfp.area_um2 / 1e3, '--', alpha=0.5, lw=1)
ax.scatter(dfp.throughput_Mhash_s, dfp.area_um2 / 1e3, s=70, zorder=3)
for _, r in dfp.iterrows():
    ax.annotate(f'U{int(r.unroll)}', (r.throughput_Mhash_s, r.area_um2 / 1e3),
                textcoords='offset points', xytext=(8, 4))
ax.set_xlabel('Throughput (Mhash/s)')
ax.set_ylabel('Stdcell area (1k $\\mu$m$^2$)')
ax.set_title('Area vs throughput (SKY130)')
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

**Figure 1 — Area vs throughput.** Throughput rises almost linearly from U1
to U4 while area grows only +38 %; past U4 the curve bends: U16 costs +115 %
area over U4 but delivers LOWER throughput (0.185 vs 0.526 Mhash/s) because
its clock is relaxed to 300 ns. The knee of this curve is U4.

In [ ]:
# Energy per hash vs throughput.
fig, ax = plt.subplots(figsize=(6.4, 4.4))
ax.plot(dfp.throughput_Mhash_s, dfp.energy_nJ, '--', alpha=0.5, lw=1)
ax.scatter(dfp.throughput_Mhash_s, dfp.energy_nJ, s=70, zorder=3)
for _, r in dfp.iterrows():
    ax.annotate(f'U{int(r.unroll)}', (r.throughput_Mhash_s, r.energy_nJ),
                textcoords='offset points', xytext=(8, 4))
ax.set_xlabel('Throughput (Mhash/s)')
ax.set_ylabel('Energy per hash (nJ)')
ax.set_title('Energy per hash vs throughput (SKY130)')
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

**Figure 2 — Energy per hash vs throughput.** This is the view that matters
for an accelerator. Energy falls steeply U1→U4 (16.94 → 4.96 nJ/hash) while
throughput more than triples (0.149 → 0.526 Mhash/s). Past U4 the curve
collapses: U16 uses **+115% area over U4 for LOWER throughput** (0.185 vs
0.526 Mhash/s) at **the same energy** (~4.95 nJ/hash) — plain unrolling past
4× does not pay. The energy-optimal-per-area point is U4.

## 7. What the data says

Read the two charts above together — this is the Pareto answer:

| UNROLL | Area (µm²) | Throughput (Mhash/s) | Energy/hash (nJ) | Power (mW) | Clock | Timing-closed |
|-------:|-----------:|---------------------:|-----------------:|-----------:|------:|:-------------:|
| 1 | 176 750 | 0.149 | 16.94 | 2.53 | 100 ns | ✅ |
| 2 | 197 497 | 0.286 | 8.98 | 2.57 | 100 ns | ✅ |
| 4 | 243 908 | 0.526 | 4.96 | 2.61 | 100 ns | ✅ |
| 16 | 524 671 | 0.185 | 4.95 | 0.92 | 300 ns | ✅ |

Three observations:

1. **U1 → U4 is the sweet spot.** Area grows modestly (+38 % U1→U4) while
   energy per hash drops **~3.4×** (16.94 → 4.96 nJ) and throughput scales
   nearly linearly (0.149 → 0.526 Mhash/s). You get most of the benefit
   for little area cost.
2. **U16 does NOT pay.** Area nearly *triples* vs U4 (+115 %), throughput
   actually *drops* (0.526 → 0.185 Mhash/s) because the 16-round critical
   path forces a 300 ns clock, and energy/hash is the same (~4.95 vs 4.96
   nJ). Plain unrolling past 4× is strictly dominated.
**Energy/hash** uses the **vectorless STA power estimate**: `P_total × cycles_per_block / f_clk`. We measure cycles/block (RTL simulation, not the closed form) as **67, 35, 19, 18, 18** for U1–U16; with the published clocks, energy/hash falls from ~16.9 nJ (U1) to ~4.96 nJ (U4) / ~4.95 nJ (U16). Energy/hash is largely clock-independent at fixed voltage; the vectorless values are **relative estimates** across variants.
3. **Power is dominated by clocking, not logic.** U1/U2/U4 all sit at
   ~2.5–2.6 mW at the same 100 ns clock despite very different cell counts;
   the clock tree + register toggling swamps the combinational differences.

> **Bottom line:** unroll to 4. U16 is strictly dominated by U4 (same energy,
> 2.15× area, 0.35× throughput). Do not unroll beyond the point where the
> input port — not the round logic — becomes the bottleneck.

### Hardened layouts (GDS → PNG)

The four hardened variants, rendered from their final GDS with KLayout
(`tools/render_layout.sh`). Standard-cell rows, the clock tree, and the I/O
pads are visible; the die grows with unroll as the round-comb logic and the
clock tree expand. Images are committed under `data/layouts/` so they render
here without re-running the flow. The cell below displays them in unroll
order (U1 → U16); each image is a **real completed GDS** — this is the
layout evidence the flow produced, not a schematic.

In [ ]:
# Display the four hardened layouts. Falls back gracefully if images absent.
for u in (1, 2, 4, 16):
    p = DATA / 'layouts' / f'u{u}.png'
    if p.exists():
        print(f'UNROLL = {u}')
        display(Image(filename=str(p), width=520))
    else:
        print(f'(no layout image for U{u} at {p})')

## 8. Conclusion

HashPareto answered one practical question with measured silicon data: *how
much should you unroll a SHA-256 core?* The answer, on SKY130 at 100–300 ns
clocks:

1. **Unrolling pays off up to 4×.** U1→U4 cuts energy per hash **~3.4×**
   (16.94 → 4.96 nJ) for only +38 % area, and throughput triples
   (0.149 → 0.526 Mhash/s). This is the knee of the Pareto curve and the
   right default for an energy-conscious accelerator.
2. **Beyond 4×, unrolling beyond 4× provides diminishing returns.** U16 needs +115 % area over U4
   for *lower* throughput (0.185 vs 0.526 Mhash/s) at the same energy
   (~4.95 nJ/hash) — because its 16-round critical path forces a 300 ns
   clock. It is never the right choice on this architecture.
3. **The method is the reusable part.** One parameter, one flow, one PDK, one
   notebook: every point is comparable, every number traces back to a committed
   metric, and the negative result (U8) is reported rather than hidden.

The full workflow — parameterized RTL, cocotb verification, LibreLane
hardening, Pareto analysis — is scripted under `tools/` and documented in
§10, so the same experiment can be repeated on any open PDK or extended to a
different compression function.

Thanks to the open-source EDA community (LibreLane, OpenROAD, Yosys, Magic,
Netgen, KLayout, Verilator, cocotb) and to the SkyWater/google SKY130
contributors — without which this notebook could not exist.


## 9. Limitations & future work

**Limitations (reported honestly):**

- **U8 (UNROLL = 8) failed at global routing** with LibreLane/OpenROAD error
  `GRT-0116` (congestion) and could not be closed within the compute budget.
  The 4-point Pareto set still demonstrates the tradeoff.
- **U16 runs at 300 ns**, not 100 ns, because its 16-round combinational path
  cannot close timing at 100 ns (no pipelining inside the round-comb loop).
- Energy is a **vectorless STA estimate** — no I/O pad power, and default
  activity factors (not traced from the cocotb vectors).
- Power/area are for the **stdcell network only** (no pads, no SRAM macros).

**Remaining work (tracked in `docs/PROJECT_STATUS.md`, phase 5):**

1. License/compliance and rubric audit — **done** (Apache-2.0 `LICENSE`,
   `NOTICE` attribution, References in §11, tool versions in §1).
2. Fork `sscs-ose/sscs-ose-code-a-chip.github.io`, copy this project
   directory to `ISSCC27/submitted_notebooks/hashpareto/`, open the PR
   before **2026-10-31, 11:59 AM Pacific**.

Optional engineering extensions (not required for submission): pipeline U16's
critical path so it can run at 100 ns; retry U8 with a larger core area /
lower placement density; add measured (SAIF-traced) power numbers.

## 10. How to reproduce everything

This notebook's §6–§7 run anywhere with `pandas` + `matplotlib` and use only
the committed `data/` files. The full flow (RTL → GDS) needs the toolchain
from §1, on a machine with ~4 GB free RAM and ~10 GB disk:

```bash
# 1. Functional verification (~2 min)
for u in 1 2 4 8 16; do
    ~/venv/bin/python verif/cocotb/run.py --sim verilator --unroll $u
done

# 2. Harden one variant (~30-45 min each)
tools/run_variant.sh 2 test_u2          # pick any UNROLL in {1,2,4,8,16}
tools/slim_run.sh flow/variants/u2/runs/test_u2

# 3. Harvest metrics + regenerate plots
~/venv/bin/python tools/collect_metrics.py
~/venv/bin/python tools/make_plots.py

# 4. (optional) render a GDS to PNG
tools/render_layout.sh flow/variants/u2/runs/test_u2/final/gds/sha_top.gds out.png 1200
```

The tables and charts in §6–§7 are computed live from `data/metrics.csv` and
`data/pareto.csv`; nothing is hand-tuned. Regenerating them on your machine
reproduces the exact numbers shown here.

**Note: running locally without pushing large artifacts.** The notebook
auto-detects its data: it looks for `data/` (submission layout) or `results/`
(development layout) in the current directory and all parent directories.
This means you can keep GB-scale `flow/runs/` and other private files on
your machine (gitignored), and **do not commit** them — only the small
committed `data/` files (CSV + PNG layouts, < 5 MB total) are needed for
the submitted package to execute.

**License:** Apache-2.0 (see `LICENSE` and `NOTICE`). The original `nano_hash`
core is also Apache-2.0, vendored unchanged under `rtl/original/`.

## 11. References

[1] NIST, **FIPS PUB 180-4**, *Secure Hash Standard (SHS)* — SHA-256 definition.

[2] `nano_hash` — reference SHA-256 core and testbenches (Apache-2.0), vendored
   read-only under `rtl/original/`. <https://github.com/chijindumu/nano_hash>

[3] LibreLane RTL-to-GDS flow (Apache-2.0) — <https://github.com/librelane/librelane>
   (v3.0.16, incl. its official Colab notebook used as the Tier-2 install template).

[4] OpenLane 2 (Apache-2.0) — <https://github.com/efabless/openlane2>;
   Yosys (ISC) — <https://github.com/YosysHQ/yosys>;
   OpenROAD/OpenSTA (Apache-2.0) — <https://github.com/The-OpenROAD-Project/OpenROAD>;
   Magic VLSI and Netgen — <http://opencircuitdesign.com/>;
   KLayout (GPL-3.0) — <https://www.klayout.de/>;
   Verilator and cocotb — <https://verilator.org/>, <https://www.cocotb.org/>.

[5] SkyWater SKY130 PDK (Apache-2.0) — <https://github.com/google/skywater-pdk>;
   `ciel` PDK manager.

[6] IEEE SSCS Code-a-Chip repository — <https://github.com/sscs-ose/sscs-ose-code-a-chip.github.io>
   (prior submissions used as format references: *Dynamically Pipelined Arithmetic*
   (VLSI'25); *CORDIC with OpenLane* (ISSCC'23); *Row-Stationary Systolic Array*
   (VLSI'24); *CircuitsDNA* / *LLMForge* (ISSCC'26)).

[7] cocotb — <https://www.cocotb.org/>; Icarus Verilog — <https://github.com/steveicarus/iverilog>.

[8] FIPS 180-4 test vectors mirrored under `data/vectors/`.

---

*HashPareto · IEEE SSCS Code-a-Chip 2027 · repo layout: this notebook +
`data/` (small committed metrics + layouts) + `rtl/` + `verif/` + `tools/` +
`flow/` (configs only; run artifacts are regenerable and not committed).*